# Legal LM from scratch: gqa_swa, 2B tokens

This notebook trains only the `gqa_swa` model (GQA + sliding-window decoder, ~197M params: d_model 1024, 16 heads / 4 KV heads, 16 layers, d_ff 2816) from random init. Training goes from step 0 to step 30518, which is 2B tokens in total (micro-batch 4 x grad-accum 16, 65536 tokens per step). The LR warms up for the first 5% of the steps and then decays with a cosine to the floor at the last step.

The 2B token file is built from the corpora in one pass (split 40/30/20/10, any shortfall goes to `high_courts`), with the validation documents left out. Building it takes long, each source is saved as a part file so a finished source is not redone. If the data ends before step 30518, the last steps go over the file again.

Training pauses itself before Kaggle's 12 h limit. Attach the output of the previous session as a dataset and run all cells again, it continues from the last checkpoint. Only attach the output of this run, not of an older one.

The tokenizer and validation set are built automatically on the first session and reused after that.

Output: `prediction_gqa_swa_2B.json` and `submission_gqa_swa_2B.zip` in `/kaggle/working/cs781_phase1/`.

### Copy the evaluator
Attach `evaluate_phase1.py` as an input to this notebook.

In [ ]:
import os
import shutil

DEST_DIR = "/kaggle/working"
os.makedirs(DEST_DIR, exist_ok=True)

print("looking for .py files in /kaggle/input")
copied_count = 0

# copy every .py file we find (the evaluator)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.py'):
            src_path = os.path.join(dirname, filename)
            dest_path = os.path.join(DEST_DIR, filename)
            shutil.copy(src_path, dest_path)
            print(f"copied {filename}")
            copied_count += 1

print("\nfiles in the working dir:")
if os.path.exists(DEST_DIR):
    files = os.listdir(DEST_DIR)
    print(f"{len(files)} files in '{DEST_DIR}'")
    for f in files:
        print(f" -> {f}")
else:
    print("ERROR: the folder does not exist")

if copied_count == 0:
    print("WARNING: no .py files found in /kaggle/input, check that the evaluator is attached")

In [ ]:
import os
import shutil
from pathlib import Path

_EVAL_BOOT = Path("/kaggle/working/cs781_phase1/eval")
_EVAL_BOOT.mkdir(parents=True, exist_ok=True)

_found = False
for dirname, _, filenames in os.walk("/kaggle/input"):
    for fn in filenames:
        if fn == "evaluate_phase1.py":
            shutil.copy(os.path.join(dirname, fn), _EVAL_BOOT / fn)
            print(f"Copied evaluator from {dirname}")
            _found = True
if not _found:
    print("WARNING: evaluate_phase1.py not found in /kaggle/input. Attach it as an input, "
          "otherwise BERTScore/ROUGE/BLEU cells will fall back to validation loss only.")

### Setup and config

In [ ]:
import time as _time0
SESSION_START = _time0.time()  # session start, used for the 12h kaggle limit

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ijson"], check=False)
for _pkg in ["bert-score", "rouge-score", "sacrebleu"]:  # deps for evaluate_phase1.py
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)

import gc, json, math, time, pickle, random, re, hashlib, itertools, zipfile, shutil
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import torch
from huggingface_hub import hf_hub_download, login

SEED = 240455
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

BASE_DIR = Path("/kaggle/working/cs781_phase1")
DATA_DIR = BASE_DIR / "data"
CHECKPOINT_DIR = BASE_DIR / "checkpoints_v3"
EVAL_DIR = BASE_DIR / "eval"
TOKENIZER_DIR = BASE_DIR / "tokenizer"
for p in [BASE_DIR, DATA_DIR, CHECKPOINT_DIR, EVAL_DIR, TOKENIZER_DIR]:
    p.mkdir(parents=True, exist_ok=True)
EVALUATOR_PATH = EVAL_DIR / "evaluate_phase1.py"

BLOCK_SIZE = 1024
MAX_SEQ_LEN = BLOCK_SIZE - 1
SRC_LEN = 640
TGT_LEN = BLOCK_SIZE - SRC_LEN
ATTN_WINDOW = 256

MICRO_BATCH = 4
GRAD_ACCUM = 16  # 4 x 16 x 1024 = 65536 tokens per step
TOTAL_TOKEN_TARGET = 2_000_000_000  # train until this many tokens in total have been seen
TARGET_TOTAL_STEP = math.ceil(TOTAL_TOKEN_TARGET / (MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE))  # 30518 steps
MAX_STEPS = TARGET_TOTAL_STEP  # whole run, step 0 to TARGET_TOTAL_STEP
NEW_STEPS_CAP = None  # optional cap on the steps of this phase
RESCHEDULE_FROM_MAX_STEPS = None  # only for resuming a run that had a different max_steps
RESUME_LR_FACTOR = None  # on resume, start the lr decay from at most this fraction of peak
EVAL_EVERY = 500
CKPT_EVERY = 250
VAL_EVAL_BLOCKS = None  # None = full val set
ADAMW_LR = 4e-4
MUON_LR = 0.02
EXTEND_LR_FACTOR = 0.3  # new phase: lr re-warms to 0.3 x peak
EXTEND_WARMUP_STEPS = 100
STOP_AFTER_TRAINING = False
TIME_BUDGET_PER_MODEL_S = None
SESSION_DEADLINE_S = 11.4 * 3600  # pause before kaggle's 12h limit
GEN_RESERVE_S = 2.5 * 3600  # time needed for generation after training
PHASE_TAG = "scratch_2B"  # tag of this run, a finished checkpoint with this tag is never extended again

NEW_TOKEN_TARGET = TOTAL_TOKEN_TARGET  # tokens to build for the training file
# 40/30/20/10 split, any shortfall goes to high_courts
EMA_DECAY = 0.995  # None = no EMA
MIXING_WEIGHTS = {"supreme_court": 0.40, "high_courts": 0.30, "acts": 0.20, "tribunals": 0.10}

MODEL_KEY = "gqa_swa"
TRAIN_KEYS = [MODEL_KEY]

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Hugging Face login successful via Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Hugging Face login successful via environment variable.")
    else:
        print("Warning: HF_TOKEN not found. Public datasets will download without authentication.")

HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

### Restore files from the attached datasets
Nothing has to be attached for the first session. In later sessions attach the output of the previous one: the furthest checkpoint, the token file (or its finished parts), the tokenizer, the validation files and any partial predictions are picked up automatically.

In [ ]:
import fnmatch

INPUT_ROOT = "/kaggle/input"

print("Attached inputs:")
for _d, _dirs, _ in os.walk(INPUT_ROOT):
    _depth = _d[len(INPUT_ROOT):].count("/")
    if _depth >= 3:
        _dirs[:] = []
    if 1 <= _depth <= 3:
        print("  " + _d)

def _find(pattern):
    hits = []
    for _d, _, _fs in os.walk(INPUT_ROOT):
        hits += [os.path.join(_d, f) for f in fnmatch.filter(_fs, pattern)]
    return sorted(hits)

def _copy_first(names, dest_dir, required=True, hint=""):
    for n in names:
        if (dest_dir / n).exists():
            continue
        h = _find(n)
        if h:
            shutil.copy(h[0], dest_dir / n); print(f"Restored {dest_dir.name}/{n}  <- {h[0]}")
        elif required:
            raise FileNotFoundError(f"{n} not found anywhere under {INPUT_ROOT}. {hint or 'Attach the output of the previous session.'}")

# nothing is needed for a fresh start, these are only picked up if an earlier session of this run saved them
# (tokenizer and validation files are rebuilt in the next cells when missing)
_copy_first(["bpe_legal_16k_v2.pkl"], TOKENIZER_DIR, required=False)
_copy_first(["val_prompts.json", "val_truth.json", "val_hashes.json", "val_ppl_texts.json"], EVAL_DIR, required=False)
_copy_first(["val_blocks.pt"], EVAL_DIR, required=False)

# token file of this run, or its finished parts
_tok_bin, _tok_json = DATA_DIR / "train_tokens_uint16_2B.bin", DATA_DIR / "train_mix_stats_2B.json"
if not (_tok_bin.exists() and _tok_json.exists()):
    _done = False
    for _d, _, _fs in os.walk(INPUT_ROOT):
        if _tok_bin.name in _fs and _tok_json.name in _fs:
            shutil.copy(os.path.join(_d, _tok_bin.name), _tok_bin); shutil.copy(os.path.join(_d, _tok_json.name), _tok_json)
            print(f"Restored the finished token file <- {_d}"); _done = True; break
    if not _done:
        _parts = DATA_DIR / "train_parts_2B"; _parts.mkdir(exist_ok=True)
        for _d, _, _fs in os.walk(INPUT_ROOT):
            if Path(_d).name != "train_parts_2B":
                continue
            for _f in _fs:
                if _f.endswith(".json") and (_f[:-5] + ".bin") in _fs and not (_parts / _f).exists():
                    shutil.copy(os.path.join(_d, _f[:-5] + ".bin"), _parts / (_f[:-5] + ".bin"))
                    shutil.copy(os.path.join(_d, _f), _parts / _f)  # json copied last so a half copied part is never used
                    print(f"Restored token-file part {_f[:-5]} <- {_d}")

# checkpoints of this run: the furthest one wins
def _pick(fname, step_of):
    best = None
    for c in _find(fname):
        st = step_of(torch.load(c, map_location="cpu", weights_only=False))
        print(f"  candidate {c}: step {st}")
        if best is None or st > best[0]:
            best = (st, c)
    return best

_last, _best = CHECKPOINT_DIR / f"{MODEL_KEY}_last.pt", CHECKPOINT_DIR / f"{MODEL_KEY}_best.pt"
if not _best.exists():
    _p = _pick(f"{MODEL_KEY}_best.pt", lambda ck: ck["step"])
    if _p:
        shutil.copy(_p[1], _best); print(f"Restored {_best.name} (step {_p[0]}) <- {_p[1]}")
if not _last.exists():
    _p = _pick(f"{MODEL_KEY}_last.pt", lambda ck: ck["global_step"])
    if _p:
        shutil.copy(_p[1], _last); print(f"Restored {_last.name} (step {_p[0]}) <- {_p[1]}")
_log = CHECKPOINT_DIR / f"{MODEL_KEY}_trainlog.json"
if not _log.exists():
    _cands = _find(f"{MODEL_KEY}_trainlog.json")
    if _cands:
        _c = max(_cands, key=lambda c: len(json.load(open(c))))
        shutil.copy(_c, _log); print(f"Restored {_log.name} <- {_c}")

if _last.exists():
    _ck = torch.load(_last, map_location="cpu", weights_only=False)
    TRAINING_DONE = bool(_ck.get("finished") and _ck.get("phase_tag") == PHASE_TAG)
    print(f"last.pt: step {_ck['global_step']} | finished={bool(_ck.get('finished'))} | phase_tag={_ck.get('phase_tag')} | "
          + ("TRAINING ALREADY FINISHED -> training is skipped, only generation runs." if TRAINING_DONE
             else ("RESUMING an unfinished run." if not _ck.get("finished")
                   else "WARNING: this is a finished checkpoint of another run, do not attach it for a fresh start.")))
    del _ck
else:
    TRAINING_DONE = False
    print("No checkpoint found -> training starts from step 0.")

if _best.exists():
    BEST_STEP = torch.load(_best, map_location="cpu", weights_only=False)["step"]
    print(f"best.pt: step {BEST_STEP}")
    # partial test predictions for this exact best checkpoint
    _pname = f"prediction_partial_{MODEL_KEY}_step{BEST_STEP}.json"
    if not (BASE_DIR / _pname).exists():
        _cands = _find(_pname)
        if _cands:
            _sizes = {c: len(json.load(open(c, encoding="utf-8"))) for c in _cands}
            _c = max(_sizes, key=_sizes.get)
            shutil.copy(_c, BASE_DIR / _pname); print(f"Restored {_pname} <- {_c} ({_sizes[_c]} predictions)")
print("Restore done.")

### Download the corpora and the test prompts

In [ ]:
CORPUS_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# train files are only needed to build the tokenizer, val set or token file
_v6_ok = (DATA_DIR / "train_tokens_uint16_2B.bin").exists() and (DATA_DIR / "train_mix_stats_2B.json").exists()
_prep_files = [TOKENIZER_DIR / "bpe_legal_16k_v2.pkl"] + [EVAL_DIR / n for n in ["val_prompts.json", "val_truth.json", "val_hashes.json", "val_ppl_texts.json"]]
NEED_CORPORA = not (_v6_ok or TRAINING_DONE) or not all(p.exists() for p in _prep_files)

downloaded_paths = {}
if NEED_CORPORA:
    for key, rel_path in CORPUS_FILES.items():
        print(f"Downloading {key}...")
        downloaded_paths[key] = hf_hub_download(repo_id=HF_REPO_ID, filename=rel_path, repo_type="dataset", token=HF_TOKEN)
        print(f"  -> {downloaded_paths[key]}")
else:
    print("Tokenizer, validation files and token file are already there (or training is finished) -> skipping the corpus download.")

test_file_path = hf_hub_download(repo_id=HF_REPO_ID, filename="Phase_1_Pretraining/Test/test.json",
                                 repo_type="dataset", token=HF_TOKEN)
print(f"Test prompts: {test_file_path}")

### Validation set
200 samples from Supreme Court and 200 from High Courts. Every document in the last 3000 of each file is hashed and skipped during training, so train and validation never overlap. A longer text per sample is also stored for validation perplexity.

In [ ]:
import ijson
from collections import deque

VAL_PROMPTS_PATH = EVAL_DIR / "val_prompts.json"
VAL_TRUTH_PATH = EVAL_DIR / "val_truth.json"
VAL_HASHES_PATH = EVAL_DIR / "val_hashes.json"
VAL_PPL_TEXTS_PATH = EVAL_DIR / "val_ppl_texts.json"

PER_SOURCE_QUOTA = {"supreme_court": 200, "high_courts": 200}
TAIL_WINDOW = 3000  # last N docs of each file are kept for validation


def doc_hash(text):
    return hashlib.md5(text.strip().encode("utf-8")).hexdigest()


if all(p.exists() for p in [VAL_PROMPTS_PATH, VAL_TRUTH_PATH, VAL_HASHES_PATH, VAL_PPL_TEXTS_PATH]):
    print("Found existing validation files. Loading from disk...")
    val_prompts = json.load(open(VAL_PROMPTS_PATH, encoding="utf-8"))
    val_truth = json.load(open(VAL_TRUTH_PATH, encoding="utf-8"))
    val_ppl_texts = json.load(open(VAL_PPL_TEXTS_PATH, encoding="utf-8"))
    val_hashes = set(json.load(open(VAL_HASHES_PATH, encoding="utf-8")))
else:
    print("Constructing 512-word prompt / 128-word continuation validation split...")
    val_prompts, val_truth, val_ppl_texts = {}, {}, {}
    val_hashes = set()

    for src, quota in PER_SOURCE_QUOTA.items():
        with open(downloaded_paths[src], "rb") as f:
            tail_records = deque(ijson.items(f, "item"), maxlen=TAIL_WINDOW)
        collected = 0
        for item in tail_records:
            text = item.get("text", "").strip()
            if not text:
                continue
            val_hashes.add(doc_hash(text))  # hash every tail doc so none of them gets trained on
            if collected >= quota:
                continue
            words = text.split()
            if len(words) >= 640:  # 512 prompt + 128 truth
                collected += 1
                sid = f"val_{src}_{collected}"
                val_prompts[sid] = {"text": " ".join(words[:512])}
                val_truth[sid] = {"text": " ".join(words[512:640])}
                val_ppl_texts[sid] = " ".join(words[:1500])
        print(f"  {src}: collected {collected}/{quota} validation samples (tail window {len(tail_records)} docs)")
        del tail_records
        gc.collect()

    json.dump(val_prompts, open(VAL_PROMPTS_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(val_truth, open(VAL_TRUTH_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(val_ppl_texts, open(VAL_PPL_TEXTS_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(sorted(val_hashes), open(VAL_HASHES_PATH, "w", encoding="utf-8"))

print(f"Validation pairs: {len(val_prompts)} | reserved doc hashes: {len(val_hashes)}")
print(f"Prompt words: {len(next(iter(val_prompts.values()))['text'].split())} | "
      f"truth words: {len(next(iter(val_truth.values()))['text'].split())}")

### BPE tokenizer
Printable ASCII plus newline and tab are always in the base vocab, and the training snippets come from random positions in each document. Training takes about an hour and the result is cached in `bpe_legal_16k_v2.pkl`.

In [ ]:
class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = {}
        self._word_cache = {}

    def _get_stats(self, vocab_freqs):
        pairs = defaultdict(int)
        for word_tuple, freq in vocab_freqs.items():
            for i in range(len(word_tuple) - 1):
                pairs[(word_tuple[i], word_tuple[i + 1])] += freq
        return pairs

    def _merge_vocab(self, pair, vocab_freqs):
        first, second = pair
        replacement = first + second
        v_out = {}
        for word_tuple, freq in vocab_freqs.items():
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_tuple.append(replacement)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            v_out[tuple(new_tuple)] = freq
        return v_out

    def train_tokenizer(self, corpus_texts, verbose=True):
        if verbose:
            print(f"Beginning BPE training to target vocab_size: {self.target_vocab_size}")
        self.vocab = {tok: idx for idx, tok in enumerate(self.special_tokens)}
        current_id = len(self.special_tokens)

        raw_word_counts = Counter()
        for doc in corpus_texts:
            raw_word_counts.update(self.pat.findall(doc))
        vocab_freqs = {tuple(word): freq for word, freq in raw_word_counts.items()}

        all_chars = set()
        for word_tuple in vocab_freqs.keys():
            all_chars.update(word_tuple)
        all_chars.update(chr(i) for i in range(32, 127))  # printable ascii always in the vocab
        all_chars.update(["\n", "\t", "\r"])
        for ch in sorted(all_chars):
            if ch not in self.vocab:
                self.vocab[ch] = current_id
                current_id += 1

        num_merges = self.target_vocab_size - len(self.vocab)
        if verbose:
            print(f"Base vocabulary size: {len(self.vocab)}. Computing {num_merges} merges...")

        for step in range(num_merges):
            pairs = self._get_stats(vocab_freqs)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            vocab_freqs = self._merge_vocab(best_pair, vocab_freqs)
            merged_token = best_pair[0] + best_pair[1]
            self.merges[best_pair] = merged_token
            self.vocab[merged_token] = current_id
            current_id += 1
            if verbose and (step + 1) % 2000 == 0:
                print(f"Completed merge {step + 1}/{num_merges} -> '{merged_token}' (ID: {current_id - 1})")

        self.inverse_vocab = {idx: tok for tok, idx in self.vocab.items()}
        self._word_cache.clear()
        if verbose:
            print(f"BPE training complete! Final vocab size: {len(self.vocab)}")

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid_merges = [p for p in pairs if p in self.merges]
            if not valid_merges:
                break
            best_pair = min(valid_merges, key=lambda p: self.vocab[self.merges[p]])
            first, second = best_pair
            merged_tok = self.merges[best_pair]
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_tuple.append(merged_tok)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        cache = self._word_cache
        ids = []
        for word in self.pat.findall(text):
            word_ids = cache.get(word)
            if word_ids is None:
                word_ids = [self.vocab.get(tok, self.unk_token_id) for tok in self._tokenize_word(word)]
                if len(cache) < 2_000_000:
                    cache[word] = word_ids
            ids.extend(word_ids)
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    def decode(self, token_ids, skip_special_tokens=True):
        tokens = []
        for tid in token_ids:
            if skip_special_tokens and tid in (self.pad_token_id, self.bos_token_id, self.eos_token_id):
                continue
            tokens.append(self.inverse_vocab.get(tid, "<unk>"))
        return "".join(tokens)

    @property
    def vocab_size(self):
        return len(self.vocab)

    def save(self, filepath):
        with open(filepath, "wb") as f:
            pickle.dump({"target_vocab_size": self.target_vocab_size, "vocab": self.vocab,
                         "inverse_vocab": self.inverse_vocab, "merges": self.merges,
                         "special_tokens": self.special_tokens}, f)
        print(f"Custom BPE tokenizer saved to: {filepath}")

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(vocab_size=data["target_vocab_size"])
        tok.vocab = data["vocab"]
        tok.inverse_vocab = data["inverse_vocab"]
        tok.merges = data["merges"]
        tok.special_tokens = data["special_tokens"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {tok.vocab_size})")
        return tok

In [ ]:
TOKENIZER_SAVE_PATH = TOKENIZER_DIR / "bpe_legal_16k_v2.pkl"
TARGET_VOCAB_SIZE = 16384


def sample_snippet(txt, rng, n=2000):
    if len(txt) <= n:
        return txt
    start = rng.randint(0, len(txt) - n)
    sp = txt.find(" ", start)
    if sp != -1 and sp - start < 100:
        start = sp + 1
    return txt[start:start + n]


if TOKENIZER_SAVE_PATH.exists():
    print("Existing BPE tokenizer found. Loading...")
    tokenizer = CustomBPETokenizer.load(TOKENIZER_SAVE_PATH)
else:
    print("Training custom BPE tokenizer on a stratified, random-position legal corpus...")
    rng = random.Random(SEED)
    TARGET_DOC_COUNTS = {"supreme_court": 8000, "high_courts": 6000, "acts": 4000, "tribunals": 2000}
    tokenizer_corpus = []
    for src_key, target_count in TARGET_DOC_COUNTS.items():
        n_got = 0
        with open(downloaded_paths[src_key], "rb") as f:
            for item in ijson.items(f, "item"):
                txt = item.get("text", "").strip()
                if not txt or doc_hash(txt) in val_hashes:
                    continue
                tokenizer_corpus.append(sample_snippet(txt, rng))
                n_got += 1
                if n_got >= target_count:
                    break
        print(f"  {src_key}: {n_got} snippets")
    print(f"Compiled {len(tokenizer_corpus)} snippets for tokenizer training.")
    tokenizer = CustomBPETokenizer(vocab_size=TARGET_VOCAB_SIZE)
    tokenizer.train_tokenizer(tokenizer_corpus, verbose=True)
    tokenizer.save(TOKENIZER_SAVE_PATH)
    del tokenizer_corpus
    gc.collect()

VOCAB_SIZE = tokenizer.vocab_size
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, tokenizer.unk_token_id
assert VOCAB_SIZE < 2 ** 16

sample_legal_text = ("In the Supreme Court of India, Criminal Appellate Jurisdiction. "
                     "Under Section 302 read with Section 34 of the Indian Penal Code, 1860, "
                     "the appellant assails the impugned judgment passed by the High Court.")
enc = tokenizer.encode(sample_legal_text, add_special_tokens=True)
dec = tokenizer.decode(enc, skip_special_tokens=True)
print(f"Sample encodes to {len(enc)} tokens; round trip OK: {dec == sample_legal_text}")
assert dec == sample_legal_text, "Round-trip tokenization check failed!"

### Token file (2B tokens)
Sources are read in the order acts, tribunals, supreme_court, high_courts, and whatever a source can't supply is added to the `high_courts` budget. Each source is saved as a part file, so an interrupted build continues where it stopped.

In [ ]:
import ijson
from torch.utils.data import Dataset, DataLoader

# fresh run, no documents are skipped
SRC_KEYS = list(MIXING_WEIGHTS)
USED_DOCS = {k: 0 for k in SRC_KEYS}
USED_TOKENS = 0
NEW_TOKEN_BUDGET = NEW_TOKEN_TARGET
EXHAUSTED = set()
print(f"Tokens to build: {NEW_TOKEN_BUDGET:,}")

EXTRA_TOKENS_PATH = DATA_DIR / "train_tokens_uint16_2B.bin"
EXTRA_STATS_PATH = DATA_DIR / "train_mix_stats_2B.json"
PARTS_DIR = DATA_DIR / "train_parts_2B"
BUILD_ORDER = ["acts", "tribunals", "supreme_court", "high_courts"]  # high_courts last, it takes whatever the others could not fill
assert set(BUILD_ORDER) == set(SRC_KEYS)
BASE_BUDGET = {k: int(w * NEW_TOKEN_BUDGET) for k, w in MIXING_WEIGHTS.items()}


def build_part(src_key, budget, already):
    part_bin, part_json = PARTS_DIR / f"{src_key}.bin", PARTS_DIR / f"{src_key}.json"
    if part_json.exists() and part_bin.exists():
        st = json.load(open(part_json))
        if st.get("target") == budget and st.get("skip_docs") == already:
            print(f"  {src_key}: reusing finished part ({st['tokens']:,} tokens)")
            return st
        print(f"  {src_key}: existing part does not match the current settings -> rebuilding")
    tmp = part_bin.with_suffix(".tmp")
    src_tokens = n_docs = passed = 0
    next_print = 10_000_000
    print(f"Processing {src_key}: skipping the first {already:,} usable documents, then reading up to {budget:,} tokens...")
    with open(tmp, "wb") as f_out, open(downloaded_paths[src_key], "rb") as f_in:
        for item in ijson.items(f_in, "item"):
            if src_tokens >= budget:
                break
            text = item.get("text", "").strip()
            if not text:
                continue
            if doc_hash(text) in val_hashes:
                continue
            if passed < already:
                passed += 1
                if passed % 50000 == 0:
                    print(f"  {src_key}: skipped {passed:,}/{already:,} documents already used")
                continue
            ids = tokenizer.encode(text, add_special_tokens=False)
            ids.append(EOS_ID)
            f_out.write(np.asarray(ids, dtype=np.uint16).tobytes())
            src_tokens += len(ids)
            n_docs += 1
            if src_tokens >= next_print:
                print(f"  {src_key}: {n_docs:,} new docs | {src_tokens:,} / {budget:,} tokens")
                next_print += 10_000_000
    os.replace(tmp, part_bin)
    st = {"tokens": src_tokens, "docs": n_docs, "target": budget, "skip_docs": already, "skipped": passed}
    json.dump(st, open(part_json, "w"))  # json last, marks the part as done
    short = max(0, budget - src_tokens)
    print(f"  -> {src_key}: {src_tokens:,} new tokens from {n_docs:,} unseen docs" + (f"  (SHORT by {short:,} tokens: the source ran out of unseen documents)" if short else ""))
    return st


NEW_DATA_STEPS = None
if EXTRA_TOKENS_PATH.exists() and EXTRA_STATS_PATH.exists():
    print(f"Found cached token file: {EXTRA_TOKENS_PATH}")
    mix_stats = json.load(open(EXTRA_STATS_PATH))
elif TRAINING_DONE:
    print("Training is already finished and the token file is not attached -> not rebuilding it (only generation runs).")
    mix_stats = {}
else:
    print("Building the 2B token file...")
    PARTS_DIR.mkdir(exist_ok=True)
    mix_stats, carried = {}, 0
    for k in BUILD_ORDER:
        budget = BASE_BUDGET[k] + (carried if k == "high_courts" else 0)
        if k in EXHAUSTED:
            print(f"  {k}: no unseen documents left -> not read again; its {budget:,}-token share goes to high_courts")
            mix_stats[k] = {"tokens": 0, "docs": 0, "target": 0, "skip_docs": USED_DOCS[k], "skipped": 0}
            if k != "high_courts":
                carried += budget
        else:
            mix_stats[k] = build_part(k, budget, USED_DOCS[k])
            if k != "high_courts":
                carried += max(0, budget - mix_stats[k]["tokens"])  # shortfall goes to high_courts
        mix_stats[k]["base_target"] = BASE_BUDGET[k]
    mix_stats["high_courts"]["took_over_from_others"] = carried
    tmp_path = EXTRA_TOKENS_PATH.with_suffix(".tmp")
    with open(tmp_path, "wb") as f_out:
        for k in ["supreme_court", "high_courts", "acts", "tribunals"]:
            if mix_stats[k]["tokens"] == 0:
                continue
            with open(PARTS_DIR / f"{k}.bin", "rb") as f_in:
                shutil.copyfileobj(f_in, f_out, 64 * 1024 * 1024)
    assert os.path.getsize(tmp_path) == 2 * sum(v["tokens"] for v in mix_stats.values()), "token file size does not match the part statistics"
    os.replace(tmp_path, EXTRA_TOKENS_PATH)
    json.dump(mix_stats, open(EXTRA_STATS_PATH, "w"), indent=2)
    shutil.rmtree(PARTS_DIR, ignore_errors=True)
    gc.collect()

if mix_stats:
    _total = sum(v["tokens"] for v in mix_stats.values())
    print("\nToken mix (planned share -> actual share):")
    for k in SRC_KEYS:
        v = mix_stats[k]
        extra = f"  | took over {v['took_over_from_others']:,} tokens that other sources could not supply" if v.get("took_over_from_others") else ""
        print(f"  {k:<14} planned {MIXING_WEIGHTS[k]:.2f} -> actual {v['tokens'] / _total:.3f}  ({v['tokens']:,} tokens, {v['docs']:,} docs){extra}")
    if _total < NEW_TOKEN_BUDGET:
        print(f"  WARNING: only {_total:,} of the planned {NEW_TOKEN_BUDGET:,} tokens exist -> the last steps will go over the file again.")
    NEW_DATA_STEPS = ((_total // BLOCK_SIZE) // MICRO_BATCH) // GRAD_ACCUM  # one epoch of the new file
    if NEW_STEPS_CAP:
        NEW_DATA_STEPS = min(NEW_DATA_STEPS, NEW_STEPS_CAP)
    _used_new = NEW_DATA_STEPS * GRAD_ACCUM * MICRO_BATCH * BLOCK_SIZE
    print(f"\nToken file: {NEW_DATA_STEPS:,} optimizer steps x {GRAD_ACCUM * MICRO_BATCH * BLOCK_SIZE:,} tokens = {_used_new:,} tokens (one epoch)")
    print(f"Training target: {TOTAL_TOKEN_TARGET:,} tokens in total = step {TARGET_TOTAL_STEP}")
    print(f"Unique tokens in the file: ~{(USED_TOKENS + _used_new) / 1e6:,.0f}M")

TOKENS_BIN_PATH = EXTRA_TOKENS_PATH


def split_block(block, mode):
    if mode == "lm":
        return block[..., :-1], block[..., 1:]
    bos = torch.full_like(block[..., :1], BOS_ID)
    return block[..., :SRC_LEN], torch.cat([bos, block[..., SRC_LEN:BLOCK_SIZE - 1]], dim=-1)


class LegalPretrainDataset(Dataset):
    def __init__(self, bin_path, block_size, mode="lm"):
        self.bin_path = str(bin_path)
        self.block_size = block_size
        self.mode = mode
        self.num_blocks = (os.path.getsize(self.bin_path) // 2) // block_size
        self._data = None

    def __len__(self):
        return self.num_blocks

    def __getitem__(self, idx):
        if self._data is None:
            self._data = np.memmap(self.bin_path, dtype=np.uint16, mode="r")
        start = idx * self.block_size
        block = torch.from_numpy(self._data[start:start + self.block_size].astype(np.int64))
        return split_block(block, self.mode)


def make_loader(mode):
    ds = LegalPretrainDataset(TOKENS_BIN_PATH, BLOCK_SIZE, mode)
    return DataLoader(ds, batch_size=MICRO_BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)


# val blocks = first BLOCK_SIZE tokens of each held-out doc
VAL_BLOCKS_PATH = EVAL_DIR / "val_blocks.pt"
if VAL_BLOCKS_PATH.exists():
    val_blocks = torch.load(VAL_BLOCKS_PATH)
else:
    _rows = []
    for sid, text in val_ppl_texts.items():
        ids = tokenizer.encode(text)
        if len(ids) >= BLOCK_SIZE:
            _rows.append(ids[:BLOCK_SIZE])
    val_blocks = torch.tensor(_rows, dtype=torch.long)
    torch.save(val_blocks, VAL_BLOCKS_PATH)
print(f"Validation blocks: {tuple(val_blocks.shape)}")

if TOKENS_BIN_PATH.exists():
    _x, _y = next(iter(make_loader("lm")))
    _s, _t = next(iter(make_loader("encdec")))
    print(f"LM batch: x{tuple(_x.shape)} y{tuple(_y.shape)} | EncDec batch: src{tuple(_s.shape)} tgt{tuple(_t.shape)}")
    del _x, _y, _s, _t

### Architectures
Only `gqa_swa` is trained here, the other classes are kept so the code stays the same as in the other notebooks. Weights are initialised N(0, 0.02) with the residual output projections scaled by 1/sqrt(2 * n_layers), and the LM head is only applied to the last position when generating.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from types import SimpleNamespace
from torch.utils.checkpoint import checkpoint as grad_ckpt


def count_parameters(model, model_name="Transformer"):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{model_name} parameter count: {params / 1e6:.2f}M")
    if not (50_000_000 <= params <= 300_000_000):
        raise ValueError(f"CONSTRAINT FAILED: {model_name} has {params / 1e6:.2f}M params, outside 50M-300M!")
    return params


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(variance + self.eps))


class SwiGLU(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.w1 = nn.Linear(in_features, hidden_features, bias=False)
        self.w2 = nn.Linear(in_features, hidden_features, bias=False)
        self.w3 = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


_ROPE = {}


def _rope_tables(head_dim, device, max_len=2048):
    key = (head_dim, str(device))
    if key not in _ROPE:
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
        t = torch.arange(max_len, device=device).float()
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        _ROPE[key] = (emb.cos(), emb.sin())
    return _ROPE[key]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    cos, sin = _rope_tables(hd, q.device)
    cos = cos[:T].to(q.dtype)[None, None]
    sin = sin[:T].to(q.dtype)[None, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


_MASK = {}


def causal_window_mask(T, window, device):
    key = (T, window, str(device))
    m = _MASK.get(key)
    if m is None:
        _MASK.clear()
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        m = (j <= i) & ((i - j) < window)  # True = can attend
        _MASK[key] = m
    return m


def attend(q, k, v, window=None):
    T = q.size(-2)
    if window is None or window >= T:
        return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=causal_window_mask(T, window, q.device))


def init_weights(model, n_layers):
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    model.apply(_init)
    for name, p in model.named_parameters():
        if name.endswith("o_proj.weight") or name.endswith("w3.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, window=None):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window = window
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rope(q, k)
        y = attend(q, k, v, self.window)
        y = y.permute(0, 2, 1, 3).contiguous().view(B, T, C)
        return self.o_proj(y)


class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.window = window
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        y = attend(q, k, v, self.window)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class _CausalLMBase(nn.Module):
    def __init__(self, vocab_size, d_model, layers, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.use_grad_ckpt = True
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(layers)
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # tied weights
        init_weights(self, len(layers))

    def forward(self, idx, labels=None):
        x = self.tok_emb(idx)
        for layer in self.layers:
            if self.use_grad_ckpt and self.training:
                x = grad_ckpt(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.final_norm(x)
        if labels is None:  # only the last position is needed when generating
            x = x[:, -1:, :]
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.reshape(-1))
        return SimpleNamespace(logits=logits, loss=loss)


# arch 1: plain decoder
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class LegalDecoderTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=12, d_ff=2048, max_seq_len=MAX_SEQ_LEN):
        super().__init__(vocab_size, d_model, [DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)], max_seq_len)


# arch 3: gqa + sliding window, every 4th layer is global
class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class LegalEfficientTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816,
                 window=ATTN_WINDOW, global_every=4, max_seq_len=MAX_SEQ_LEN):
        layers = [EfficientDecoderBlock(d_model, n_heads, n_kv_heads, d_ff,
                                        window=None if (i + 1) % global_every == 0 else window)
                  for i in range(n_layers)]
        super().__init__(vocab_size, d_model, layers, max_seq_len)


# arch 4: gru / local attention hybrid
class HybridDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, layer_idx, window):
        super().__init__()
        self.use_attn = (layer_idx % 2 == 0)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.mixer = CausalSelfAttention(d_model, n_heads, window=window) if self.use_attn \
            else nn.GRU(d_model, d_model, batch_first=True)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        h = self.norm1(x)
        h = self.mixer(h) if self.use_attn else self.mixer(h)[0]
        x = x + h
        return x + self.ffn(self.norm2(x))


class LegalHybridRNNTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=12, d_ff=2048,
                 window=ATTN_WINDOW, max_seq_len=MAX_SEQ_LEN):
        layers = [HybridDecoderBlock(d_model, n_heads, d_ff, i, window) for i in range(n_layers)]
        super().__init__(vocab_size, d_model, layers, max_seq_len)


# arch 2: encoder-decoder
class LegalEncDecTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, num_encoder_layers=6, num_decoder_layers=6,
                 d_ff=2048, max_seq_len=BLOCK_SIZE, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_encoder_layers, norm=nn.LayerNorm(d_model),
                                             enable_nested_tensor=False)
        dec_layer = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout=dropout, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_decoder_layers, norm=nn.LayerNorm(d_model))
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        init_weights(self, num_encoder_layers + num_decoder_layers)

    def encode(self, src_idx):
        pos = torch.arange(src_idx.size(1), device=src_idx.device)
        return self.encoder(self.tok_emb(src_idx) + self.pos_emb(pos)[None])

    def forward(self, src_idx, tgt_idx=None, labels=None, memory=None):
        if memory is None:
            memory = self.encode(src_idx)
        if tgt_idx is None:
            tgt_idx = labels
        T = tgt_idx.size(1)
        pos = torch.arange(T, device=tgt_idx.device)
        tgt_emb = self.tok_emb(tgt_idx) + self.pos_emb(pos)[None]
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=tgt_idx.device)
        dec = self.decoder(tgt_emb, memory, tgt_mask=mask)
        loss = None
        if labels is not None:
            logits = self.lm_head(dec)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        else:
            logits = self.lm_head(dec[:, -1:, :])
        return SimpleNamespace(logits=logits, loss=loss)


# registry
MODEL_REGISTRY = {
    "decoder": (LegalDecoderTransformer, dict(vocab_size=VOCAB_SIZE, d_model=768, n_heads=12, n_layers=12, d_ff=2048)),
    "gqa_swa": (LegalEfficientTransformer, dict(vocab_size=VOCAB_SIZE, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816)),
    "hybrid": (LegalHybridRNNTransformer, dict(vocab_size=VOCAB_SIZE, d_model=768, n_heads=12, n_layers=12, d_ff=2048)),
    "encdec": (LegalEncDecTransformer, dict(vocab_size=VOCAB_SIZE, d_model=512, n_heads=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048)),
}


def build_model(key):
    cls, cfg = MODEL_REGISTRY[key]
    return cls(**cfg)

In [ ]:
# param count check, built on cpu and freed again
for _key in TRAIN_KEYS:
    _m = build_model(_key)
    count_parameters(_m, MODEL_REGISTRY[_key][0].__name__)
    del _m
gc.collect()

### Training
Muon for the weight matrices, AdamW for embeddings, the tied head, norms and biases. Both optimizers are unscaled before clipping and both go through `scaler.step`. `<key>_last.pt` is a resumable checkpoint and `<key>_best.pt` stores the best weights with the model class and config.

In [ ]:
from torch.amp import GradScaler
from torch.optim.lr_scheduler import LambdaLR


@torch.no_grad()
def newton_schulz_orthogonalize(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G / (G.norm() + eps)
    transposed = G.size(0) > G.size(1)
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = A @ X
        X = a * X + b * B + c * (A @ B)
    return X.T if transposed else X


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95):
        super().__init__(params, dict(lr=lr, momentum=momentum))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)
                buf = state["momentum_buffer"]
                buf.mul_(group["momentum"]).add_(p.grad)
                O = newton_schulz_orthogonalize(buf)
                scale = max(1.0, math.sqrt(p.size(0) / p.size(1)))
                p.sub_(O, alpha=group["lr"] * scale)


def setup_optimizers(model, adamw_lr=ADAMW_LR, muon_lr=MUON_LR):
    muon_params, adamw_decay, adamw_no_decay = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 2 and "emb" not in name and "lm_head" not in name:
            muon_params.append(p)
        elif p.ndim >= 2:
            adamw_decay.append(p)
        else:
            adamw_no_decay.append(p)
    opt_muon = Muon(muon_params, lr=muon_lr) if muon_params else None
    opt_adamw = torch.optim.AdamW(
        [{"params": adamw_decay, "weight_decay": 0.05}, {"params": adamw_no_decay, "weight_decay": 0.0}],
        lr=adamw_lr, betas=(0.9, 0.95))
    print(f"Optimizer routing: {len(muon_params)} matrices -> Muon, "
          f"{len(adamw_decay) + len(adamw_no_decay)} tensors -> AdamW")
    return opt_muon, opt_adamw


def model_mode(model):
    return "encdec" if isinstance(model, LegalEncDecTransformer) else "lm"


def forward_loss(model, batch):
    inp, tgt = batch
    return model(inp, labels=tgt).loss


@torch.no_grad()
def evaluate_loss(model, blocks_tensor, batch_size=8, max_blocks=None, device=DEVICE):
    was_training = model.training
    model.eval()
    mode = model_mode(model)
    blocks = blocks_tensor if max_blocks is None else blocks_tensor[:max_blocks]
    tot, n = 0.0, 0
    for i in range(0, len(blocks), batch_size):
        b = blocks[i:i + batch_size].to(device)
        inp, tgt = split_block(b, mode)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            loss = model(inp, labels=tgt).loss
        ntok = tgt[..., 1:].numel() if mode == "encdec" else tgt.numel()
        tot += loss.item() * ntok
        n += ntok
    if was_training:
        model.train()
    return tot / max(n, 1)


def _atomic_save(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _model_payload(key, model, step, val_loss):
    cls, cfg = MODEL_REGISTRY[key]
    return {"model_key": key, "model_class": cls.__name__, "config": cfg,
            "state_dict": model.state_dict(), "step": step, "val_loss": val_loss}


def _ema_update(ema, model, decay):
    with torch.no_grad():
        for e, p in zip(ema, model.parameters()):
            e.mul_(decay).add_(p.detach(), alpha=1.0 - decay)


def _eval_with_ema(model, ema, blocks, max_blocks, on_eval=None):
    params = list(model.parameters())
    backup = [p.detach().clone() for p in params]
    with torch.no_grad():
        for p, e in zip(params, ema):
            p.copy_(e)
    try:
        vl = evaluate_loss(model, blocks, max_blocks=max_blocks)
        if on_eval is not None:
            on_eval(vl)
    finally:
        with torch.no_grad():
            for p, b in zip(params, backup):
                p.copy_(b)
        del backup
    return vl


def train_model(key, max_steps=None, grad_accum=None, eval_every=None, ckpt_every=None, time_budget_s=None):
    auto_steps = max_steps is None and MAX_STEPS is None
    max_steps = max_steps or MAX_STEPS or 1_000_000_000
    grad_accum = grad_accum or GRAD_ACCUM
    eval_every = eval_every or EVAL_EVERY
    ckpt_every = ckpt_every or CKPT_EVERY
    time_budget_s = time_budget_s if time_budget_s is not None else TIME_BUDGET_PER_MODEL_S

    last_path = CHECKPOINT_DIR / f"{key}_last.pt"
    best_path = CHECKPOINT_DIR / f"{key}_best.pt"
    log_path = CHECKPOINT_DIR / f"{key}_trainlog.json"
    use_cuda = DEVICE.type == "cuda"

    model = build_model(key).to(DEVICE)
    opt_muon, opt_adamw = setup_optimizers(model)
    scaler = GradScaler("cuda", enabled=use_cuda)

    warmup = max(1, int(0.05 * max_steps))

    def _cosine_factor(step, total, warm):
        if step < warm:
            return float(step + 1) / float(warm)
        progress = float(step - warm) / float(max(1, total - warm))
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

    # lr override used when resuming or extending a run
    sched = {"s0": None, "f0": None, "fp": None, "warm": 0}

    def lr_lambda(step):
        if sched["s0"] is not None and step >= sched["s0"]:
            fp = sched["fp"] if sched.get("fp") is not None else sched["f0"]
            warm = sched.get("warm", 0)
            t = step - sched["s0"]
            if t < warm:
                return sched["f0"] + (fp - sched["f0"]) * float(t) / float(warm)
            progress = min(1.0, float(t - warm) / float(max(1, max_steps - sched["s0"] - warm)))
            return 0.05 + (fp - 0.05) * 0.5 * (1.0 + math.cos(math.pi * progress))
        return _cosine_factor(step, max_steps, warmup)

    sch_adamw = LambdaLR(opt_adamw, lr_lambda)
    sch_muon = LambdaLR(opt_muon, lr_lambda) if opt_muon else None

    global_step, best_val, history = 0, float("inf"), []
    if last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        if ck.get("finished") and (ck.get("phase_tag") == PHASE_TAG or ck["global_step"] >= max_steps):
            print(f"[{key}] already finished (step {ck['global_step']}, best val loss {ck['best_val']:.4f}). Skipping.")
            return {"key": key, "steps": ck["global_step"], "best_val_loss": ck["best_val"], "finished": True}
        if ck.get("finished"):
            # run already finished: last.pt has no optimizer state, so start from best.pt weights with fresh optimizers
            assert best_path.exists(), f"{best_path} is missing - cannot extend a finished run without its weights."
            bk = torch.load(best_path, map_location="cpu", weights_only=False)
            model.load_state_dict(bk["state_dict"])
            global_step = bk["step"]
            if auto_steps:
                assert NEW_DATA_STEPS, "the new token file is not available - cannot start the new phase"
                max_steps = global_step + NEW_DATA_STEPS
                print(f"[{key}] NEW PHASE: {NEW_DATA_STEPS:,} steps on the new token file -> step {global_step} to step {max_steps}")
            best_val = evaluate_loss(model, val_blocks)
            print(f"[{key}] starting weights: step {global_step}, full-validation loss {best_val:.4f} (ppl {math.exp(best_val):.2f})")
            sched.update(s0=global_step, f0=0.05, fp=EXTEND_LR_FACTOR, warm=EXTEND_WARMUP_STEPS)
            for _s in (sch_adamw, sch_muon):
                if _s is not None:
                    _s.last_epoch = global_step - 1
                    _s.step()
            if log_path.exists():
                history = json.load(open(log_path))
            print(f"[{key}] EXTENDING finished run: warm start from best.pt (step {global_step}, val loss {best_val:.4f}), fresh optimizer state")
            print(f"[{key}] LR: {sch_adamw.get_last_lr()[0]:.2e} at step {global_step} -> re-warm to {EXTEND_LR_FACTOR * ADAMW_LR:.2e} over "
                  f"{EXTEND_WARMUP_STEPS} steps -> cosine to {0.05 * ADAMW_LR:.2e} at step {max_steps}")
            del bk
        else:
            model.load_state_dict(ck["model"])
            opt_adamw.load_state_dict(ck["opt_adamw"])
            if opt_muon and ck.get("opt_muon") is not None:
                opt_muon.load_state_dict(ck["opt_muon"])
            sch_adamw.load_state_dict(ck["sch_adamw"])
            if sch_muon and ck.get("sch_muon") is not None:
                sch_muon.load_state_dict(ck["sch_muon"])
            scaler.load_state_dict(ck["scaler"])
            global_step, best_val = ck["global_step"], ck["best_val"]
            if ck.get("sched_anchor"):
                sched.update(ck["sched_anchor"])
            elif RESCHEDULE_FROM_MAX_STEPS and RESCHEDULE_FROM_MAX_STEPS != max_steps and global_step < max_steps:
                sched["s0"] = global_step
                sched["f0"] = _cosine_factor(global_step, RESCHEDULE_FROM_MAX_STEPS, max(1, int(0.05 * RESCHEDULE_FROM_MAX_STEPS)))
                if RESUME_LR_FACTOR is not None:
                    sched["f0"] = min(sched["f0"], RESUME_LR_FACTOR)
                print(f"[{key}] LR schedule re-anchored: factor {sched['f0']:.3f} at step {global_step} -> 0.05 at step {max_steps}")
            if auto_steps:
                assert NEW_DATA_STEPS and sched["s0"] is not None, "cannot resume this phase: new token file or phase anchor missing"
                max_steps = sched["s0"] + NEW_DATA_STEPS
                print(f"[{key}] phase started at step {sched['s0']}; it ends at step {max_steps}")
            if log_path.exists():
                history = json.load(open(log_path))
            print(f"[{key}] resumed from step {global_step} (best val loss so far {best_val:.4f})")
        del ck

    mode = "encdec" if key == "encdec" else "lm"
    loader = make_loader(mode)
    data_iter = iter(loader)
    _data_start = sched["s0"] if sched["s0"] is not None else 0
    if global_step > _data_start:
        # resumed inside the new-data phase, skip the batches already used (shuffle order comes from SEED)
        _skip = (global_step - _data_start) * grad_accum
        print(f"[{key}] fast-forwarding the data order past {_skip:,} micro-batches already used ...")
        for _ in range(_skip):
            try:
                next(data_iter)
            except StopIteration:
                data_iter = iter(loader)
        print(f"[{key}] data order positioned; continuing training.")
    ema = [p.detach().clone() for p in model.parameters()] if EMA_DECAY else None
    if ema is not None:
        print(f"[{key}] EMA of the weights enabled (decay {EMA_DECAY}); it is validated next to the raw weights.")
    model.train()
    t_start, t_log, tok_since_log, loss_since_log, steps_since_log = time.time(), time.time(), 0, 0.0, 0
    stopped_early = False

    def save_last(finished=False):
        if finished:
            _atomic_save({"finished": True, "global_step": global_step, "best_val": best_val, "phase_tag": PHASE_TAG}, last_path)
            return
        _atomic_save({
            "model": model.state_dict(), "opt_adamw": opt_adamw.state_dict(),
            "opt_muon": opt_muon.state_dict() if opt_muon else None,
            "sch_adamw": sch_adamw.state_dict(), "sch_muon": sch_muon.state_dict() if sch_muon else None,
            "scaler": scaler.state_dict(), "global_step": global_step, "best_val": best_val,
            "sched_anchor": dict(sched) if sched["s0"] is not None else None,
        }, last_path)
        json.dump(history, open(log_path, "w"))

    def validate(final=False):
        nonlocal best_val
        mb = None if final else VAL_EVAL_BLOCKS
        vl = evaluate_loss(model, val_blocks, max_blocks=mb)
        rec = {"step": global_step, "val_loss": vl, "final": final}
        msg = f"[{key}] step {global_step} | val loss {vl:.4f} | ppl {math.exp(vl):.2f}"
        if vl < best_val:
            best_val = vl
            _atomic_save(_model_payload(key, model, global_step, vl), best_path)
            msg += "  * new best"
        if ema is not None:
            _hit = []

            def _save_if_best(v):
                nonlocal best_val
                if v < best_val:
                    best_val = v
                    _atomic_save(_model_payload(key, model, global_step, v), best_path)
                    _hit.append(True)

            vl_e = _eval_with_ema(model, ema, val_blocks, mb, on_eval=_save_if_best)
            rec["ema_val_loss"] = vl_e
            msg += f" | EMA val loss {vl_e:.4f} (ppl {math.exp(vl_e):.2f})" + ("  * new best (EMA weights)" if _hit else "")
        history.append(rec)
        print(msg)
        model.train()
        return vl

    while global_step < max_steps:
        for _ in range(grad_accum):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(loader)
                batch = next(data_iter)
            batch = [t.to(DEVICE, non_blocking=True) for t in batch]
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_cuda):
                loss = forward_loss(model, batch) / grad_accum
            scaler.scale(loss).backward()
            loss_since_log += loss.item()
            tok_since_log += batch[0].numel() + (batch[1].numel() if mode == "encdec" else 0)

        scaler.unscale_(opt_adamw)
        if opt_muon:
            scaler.unscale_(opt_muon)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt_adamw)
        if opt_muon:
            scaler.step(opt_muon)
        scaler.update()
        opt_adamw.zero_grad(set_to_none=True)
        if opt_muon:
            opt_muon.zero_grad(set_to_none=True)
        sch_adamw.step()
        if sch_muon:
            sch_muon.step()
        global_step += 1
        steps_since_log += 1
        if ema is not None:
            _ema_update(ema, model, EMA_DECAY)

        if global_step % 20 == 0:
            dt = time.time() - t_log
            print(f"[{key}] step {global_step}/{max_steps} | train loss {loss_since_log / steps_since_log:.4f} | "
                  f"lr {sch_adamw.get_last_lr()[0]:.2e} | {tok_since_log / max(dt, 1e-6):,.0f} tok/s")
            t_log, tok_since_log, loss_since_log, steps_since_log = time.time(), 0, 0.0, 0
        if global_step % eval_every == 0 and global_step < max_steps:
            validate()
        if global_step % ckpt_every == 0 and global_step < max_steps:
            save_last()
        if time_budget_s is not None and (time.time() - t_start) > time_budget_s and global_step < max_steps:
            print(f"[{key}] time budget reached at step {global_step}; saving resumable checkpoint.")
            save_last()
            stopped_early = True
            break

    summary = {"key": key, "steps": global_step, "finished": not stopped_early}
    if not stopped_early:
        validate(final=True)
        history_final = history[-1]["val_loss"]
        # save the final weights too, best.pt can be from an earlier step
        _atomic_save(_model_payload(key, model, global_step, history_final), CHECKPOINT_DIR / f"{key}_final_raw.pt")
        if ema is not None:
            _eval_with_ema(model, ema, val_blocks, None,
                           on_eval=lambda v: _atomic_save(_model_payload(key, model, global_step, v), CHECKPOINT_DIR / f"{key}_final_ema.pt"))
        summary.update(best_val_loss=best_val, final_val_loss=history_final)
        json.dump(history, open(log_path, "w"))
        save_last(finished=True)
    else:
        summary["best_val_loss"] = best_val

    del model, opt_adamw, opt_muon, loader, data_iter
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
    return summary

### Train the model
If the session stops, run the cells above again and then this one. A finished model is skipped and an unfinished one resumes from its last checkpoint.

In [ ]:
training_summaries = {}
for key in TRAIN_KEYS:
    print("\n" + "=" * 70 + f"\nTraining {key}\n" + "=" * 70)
    _remaining = max(600.0, SESSION_DEADLINE_S - (time.time() - SESSION_START))
    print(f"Session time used so far: {(time.time() - SESSION_START) / 3600:.2f} h -> training may run up to {_remaining / 3600:.2f} h in this session")
    training_summaries[key] = train_model(key, time_budget_s=_remaining)
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
print(json.dumps(training_summaries, indent=2))


### Stop here if training isn't finished
If the time budget paused training, the next cell stops the run so nothing below runs on a half trained model. Save the version, attach its output to a new session and run again.

In [ ]:
_unfinished = [k for k, v in training_summaries.items() if not v.get("finished")]
if _unfinished:
    raise RuntimeError(f"Training of {_unfinished} paused at the time budget (checkpoint saved in {CHECKPOINT_DIR}). "
                       "Stopping on purpose: save this version, attach its output to a new session, and run again to resume.")
if STOP_AFTER_TRAINING:
    raise RuntimeError("STOP_AFTER_TRAINING is True: training is finished and saved (checkpoints_v3/). Stopping on purpose - "
                       "set STOP_AFTER_TRAINING = False and run again for generation / evaluation.")
_elapsed = time.time() - SESSION_START
if _elapsed + GEN_RESERVE_S > 11.7 * 3600:
    raise RuntimeError(f"Training is finished and saved, but only {(12 * 3600 - _elapsed) / 3600:.1f} h of this session are left and generation needs about {GEN_RESERVE_S / 3600:.1f} h. "
                       "Stopping on purpose: save this version, attach its output to a NEW session and run again - training is skipped and only the decoding sweep, "
                       "test inference and the zip are produced.")
print("Training finished for all models -> continuing with generation / evaluation.")


### Generation and evaluation helpers
`<pad>`, `<unk>` and `<s>` are never generated and `</s>` is blocked until there are enough words. The repetition penalty only looks at generated tokens. One extra word is generated so the 128th word is never a cut-off BPE piece.

In [ ]:
def filter_logits(logits, top_k=0, top_p=1.0):
    if top_k and top_k > 0:
        kth_best = torch.topk(logits, min(top_k, logits.size(-1)))[0][..., -1, None]
        logits = logits.masked_fill(logits < kth_best, float("-inf"))
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        remove_sorted = cum_probs > top_p
        remove_sorted[..., 1:] = remove_sorted[..., :-1].clone()
        remove_sorted[..., 0] = False
        remove = remove_sorted.scatter(-1, sorted_idx, remove_sorted)
        logits = logits.masked_fill(remove, float("-inf"))
    return logits


@torch.no_grad()
def generate_continuation_custom(model, tokenizer, prompt_text, target_words=128, max_new_tokens=320,
                                 temperature=0.4, top_p=0.92, top_k=50, repetition_penalty=1.15,
                                 max_prompt_tokens=None, device=DEVICE):
    model.eval()
    is_enc_dec = isinstance(model, LegalEncDecTransformer)
    use_cuda = device.type == "cuda"

    prompt_tokens = tokenizer.encode(prompt_text, add_special_tokens=False)
    if is_enc_dec:
        limit = SRC_LEN if max_prompt_tokens is None else min(SRC_LEN, max_prompt_tokens)
        max_new_tokens = min(max_new_tokens, TGT_LEN - 1)
    else:
        limit = model.max_seq_len - max_new_tokens
        if max_prompt_tokens is not None:
            limit = min(limit, max_prompt_tokens)
    prompt_tokens = prompt_tokens[-limit:]
    prompt_tensor = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    blocked = torch.tensor([PAD_ID, UNK_ID, BOS_ID], dtype=torch.long, device=device)
    need_words = target_words + 1
    generated = []
    ctx = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_cuda)

    memory, tgt_tokens, curr_seq = None, [BOS_ID], prompt_tensor
    if is_enc_dec:
        with ctx:
            memory = model.encode(prompt_tensor)

    for _ in range(max_new_tokens):
        with ctx:
            if is_enc_dec:
                tgt = torch.tensor([tgt_tokens], dtype=torch.long, device=device)
                out = model(prompt_tensor, tgt_idx=tgt, memory=memory)
            else:
                if curr_seq.size(1) > model.max_seq_len:
                    curr_seq = curr_seq[:, -model.max_seq_len:]
                out = model(curr_seq)
        logits = out.logits[0, -1, :].float() / max(temperature, 1e-5)

        if repetition_penalty != 1.0 and generated:
            seen = torch.tensor(sorted(set(generated)), dtype=torch.long, device=device)
            prev = logits[seen]
            logits[seen] = torch.where(prev < 0, prev * repetition_penalty, prev / repetition_penalty)

        logits[blocked] = float("-inf")
        current_words = len(tokenizer.decode(generated, skip_special_tokens=True).split())
        if current_words < need_words:
            logits[EOS_ID] = float("-inf")

        probs = F.softmax(filter_logits(logits.unsqueeze(0), top_k=top_k, top_p=top_p).squeeze(0), dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        if next_token == EOS_ID:
            break
        generated.append(next_token)
        if is_enc_dec:
            tgt_tokens.append(next_token)
        else:
            curr_seq = torch.cat([curr_seq, torch.tensor([[next_token]], device=device)], dim=1)
        if len(tokenizer.decode(generated, skip_special_tokens=True).split()) >= need_words:
            break

    words = tokenizer.decode(generated, skip_special_tokens=True).split()[:target_words]
    while len(words) < target_words:
        words.append("the")
    return " ".join(words)


def load_trained(key, device=DEVICE):
    payload = torch.load(CHECKPOINT_DIR / f"{key}_best.pt", map_location="cpu", weights_only=False)
    cls = {c.__name__: c for c, _ in MODEL_REGISTRY.values()}[payload["model_class"]]
    model = cls(**payload["config"])
    model.load_state_dict(payload["state_dict"])
    return model.to(device).eval(), payload


def parse_metrics(text):
    def grab(pat):
        m = re.search(pat, text, re.I)
        return float(m.group(1)) if m else None
    return {
        "bertscore_f1": grab(r"bert\s*score[^\n\d-]*?(?:f1)?[^\n\d-]*?(-?\d+\.\d+)"),
        "rougeL_f1": grab(r"rouge[-_ ]?l[^\n\d-]*?(?:f1)?[^\n\d-]*?(-?\d+\.\d+)"),
        "bleu4": grab(r"bleu[-_ ]?4?[^\n\d-]*?(-?\d+\.\d+)"),
    }


def score_with_evaluator(truth_path, pred_path):
    if not EVALUATOR_PATH.exists():
        print("  (evaluate_phase1.py missing - skipping BERTScore/ROUGE/BLEU)")
        return {"bertscore_f1": None, "rougeL_f1": None, "bleu4": None}
    proc = subprocess.run([sys.executable, str(EVALUATOR_PATH), "--truth", str(truth_path), "--prediction", str(pred_path)],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        print("  EVALUATOR FAILED:\n" + proc.stderr[-1500:])
        return {"bertscore_f1": None, "rougeL_f1": None, "bleu4": None}
    metrics = parse_metrics(proc.stdout)
    if any(v is None for v in metrics.values()):
        print("  Could not parse every metric; raw evaluator output:\n" + proc.stdout[-1500:])
    return metrics


def primary_score(metrics):
    for k in ("bertscore_f1", "rougeL_f1", "bleu4"):
        if metrics.get(k) is not None:
            return metrics[k]
    return None


def run_eval(model, ids, gen_kwargs, tag):
    preds, t0 = {}, time.time()
    for i, pid in enumerate(ids):
        preds[pid] = {"text": generate_continuation_custom(model, tokenizer, val_prompts[pid]["text"], **gen_kwargs)}
        if (i + 1) % 25 == 0:
            print(f"  [{tag}] {i + 1}/{len(ids)} in {time.time() - t0:.0f}s")
    assert all(len(v["text"].split()) == 128 for v in preds.values()), "a prediction is not exactly 128 words"
    pred_path, truth_path = EVAL_DIR / f"pred_{tag}.json", EVAL_DIR / f"truth_{tag}.json"
    json.dump(preds, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump({pid: val_truth[pid] for pid in ids}, open(truth_path, "w", encoding="utf-8"), ensure_ascii=False)
    metrics = score_with_evaluator(truth_path, pred_path)
    metrics["gen_time_s"] = round(time.time() - t0, 1)
    return metrics


ALL_VAL_IDS = list(val_prompts.keys())
EVAL_IDS = ALL_VAL_IDS[::4]  # 100 prompts
SWEEP_IDS = ALL_VAL_IDS[::20]  # 20 prompts for the sweeps
print(f"Eval subset: {len(EVAL_IDS)} prompts | sweep subset: {len(SWEEP_IDS)} prompts")

### Validation loss and perplexity

In [ ]:
val_results = {}
for key in TRAIN_KEYS:
    model, payload = load_trained(key)
    vl = evaluate_loss(model, val_blocks)
    val_results[key] = {"val_loss": vl, "val_ppl": math.exp(vl), "step": payload["step"],
                        "params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print(f"{'model':<10} {'params(M)':>10} {'best step':>10} {'val loss':>9} {'val ppl':>9}")
for k, v in val_results.items():
    note = "  (continuation-only, not comparable)" if k == "encdec" else ""
    print(f"{k:<10} {v['params_M']:>10.2f} {v['step']:>10} {v['val_loss']:>9.4f} {v['val_ppl']:>9.2f}{note}")
json.dump(val_results, open(EVAL_DIR / "val_results.json", "w"), indent=2)

### Decoding sweep (20 prompts)

In [ ]:
SWEEP_CONFIGS = [
    {"name": "Greedy-like", "temperature": 0.2, "top_p": 0.90, "repetition_penalty": 1.10},
    {"name": "Balanced", "temperature": 0.4, "top_p": 0.92, "repetition_penalty": 1.15},
    {"name": "Creative", "temperature": 0.7, "top_p": 0.95, "repetition_penalty": 1.20},
    {"name": "No-rep-penalty", "temperature": 0.4, "top_p": 0.92, "repetition_penalty": 1.00},
]
causal_keys = [k for k in TRAIN_KEYS if k != "encdec"]
sweep_key = MODEL_KEY
print(f"Sweeping decoding on '{sweep_key}' (the model trained in this notebook)")
model, _ = load_trained(sweep_key)

sweep_results = {}
for cfg in SWEEP_CONFIGS:
    kw = {k: v for k, v in cfg.items() if k != "name"}
    m = run_eval(model, SWEEP_IDS, kw, f"sweep_{cfg['name']}")
    sweep_results[cfg["name"]] = {"gen_kwargs": kw, **m}
    print(f"[{cfg['name']}] {m}")

scored = {n: primary_score(r) for n, r in sweep_results.items() if primary_score(r) is not None}
best_cfg_name = max(scored, key=scored.get) if scored else "Balanced"
BEST_GEN_KWARGS = dict(sweep_results[best_cfg_name]["gen_kwargs"], top_k=50)
print(f"\nBest decoding config: {best_cfg_name} -> {BEST_GEN_KWARGS}")
json.dump(sweep_results, open(EVAL_DIR / "decoding_sweep.json", "w"), indent=2)
del model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

### Model used for the submission

In [ ]:
BEST_MODEL_KEY = MODEL_KEY
print("Model used for the test submission:", BEST_MODEL_KEY, "| decoding:", BEST_GEN_KWARGS)


### Test inference and submission zip
Uses the best decoding config from the sweep. Partial predictions are saved every 25 prompts so generation can continue in another session.

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_raw = json.load(f)
if isinstance(test_raw, dict):
    test_items = [(str(k), v) for k, v in test_raw.items()]
else:
    test_items = [(str(r.get("id", i)), r) for i, r in enumerate(test_raw)]
print(f"Official test examples: {len(test_items)} | first id: '{test_items[0][0]}'")

best_model, _best_payload = load_trained(BEST_MODEL_KEY)
# partials are tied to the checkpoint step so old text is never reused
PARTIAL_PATH = BASE_DIR / f"prediction_partial_{MODEL_KEY}_step{_best_payload['step']}.json"
OFFICIAL_PRED_PATH = BASE_DIR / f"prediction_{MODEL_KEY}_2B.json"
TEST_PREDICTIONS = json.load(open(PARTIAL_PATH, encoding="utf-8")) if PARTIAL_PATH.exists() else {}
print(f"Generating with the step-{_best_payload['step']} checkpoint | resuming with {len(TEST_PREDICTIONS)} predictions already done")
t0 = time.time()
for i, (ex_id, rec) in enumerate(test_items):
    if ex_id in TEST_PREDICTIONS:
        continue
    TEST_PREDICTIONS[ex_id] = {"text": generate_continuation_custom(best_model, tokenizer, rec["text"], **BEST_GEN_KWARGS)}
    if len(TEST_PREDICTIONS) % 25 == 0:
        json.dump(TEST_PREDICTIONS, open(PARTIAL_PATH, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"Test generation: {len(TEST_PREDICTIONS)}/{len(test_items)} | {time.time() - t0:.0f}s")

json.dump(TEST_PREDICTIONS, open(OFFICIAL_PRED_PATH, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

# sanity checks
assert len(TEST_PREDICTIONS) == len(test_items), "Prediction count does not match test set size!"
assert set(TEST_PREDICTIONS) == {k for k, _ in test_items}, "Test ID mismatch!"
bad = [k for k, v in TEST_PREDICTIONS.items() if len(v["text"].split()) != 128]
assert not bad, f"{len(bad)} predictions are not exactly 128 words (e.g. {bad[:3]})"
print("Integrity verified: 1:1 ID match, every output is exactly 128 words.")

SUBMISSION_ZIP_PATH = BASE_DIR / f"submission_{MODEL_KEY}_2B.zip"
with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(OFFICIAL_PRED_PATH, arcname="prediction.json")
with zipfile.ZipFile(SUBMISSION_ZIP_PATH) as zf:
    assert zf.namelist() == ["prediction.json"], "prediction.json must be at the zip root"
print(f"Created {SUBMISSION_ZIP_PATH}")

### Reuse later
Every `<key>_best.pt` has `state_dict`, `model_class`, `config`, `step` and `val_loss`. Use `load_trained(key)` or
`cls = MODEL_REGISTRY[key][0]; m = cls(**payload["config"]); m.load_state_dict(payload["state_dict"])`.
Back up `checkpoints_v3/`, `tokenizer/bpe_legal_16k_v2.pkl` and `eval/` before the session ends.